# Week 3 확장 — ePillID 5종 알약 분류

객체 탐지 다음 단계로, 잘라낸 알약 한 장을 5개 종류 중 하나로 분류한다. 공개 ePillID 데이터의 작은 부분집합을 사용해 YOLOv8n-cls를 fine-tuning하고 별도 test 이미지로 평가한다.

> 연구·학습용 실험이며 약품 복용이나 의료 판단에 사용할 수 없다.

## 1. 환경 설정
Colab GPU(T4)를 권장한다. 결과 재현을 위해 seed를 고정한다.

In [ ]:
!pip -q install "ultralytics>=8.3,<9"

import json
import random
import shutil
import urllib.request
import zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from ultralytics import YOLO, __version__ as ultralytics_version

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

WORK_DIR = Path('/content/week3_classification')
DATASET_DIR = WORK_DIR / 'epillid_5class'
OUTPUT_DIR = WORK_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

print('device:', DEVICE)
print('ultralytics:', ultralytics_version)

## 2. ePillID에서 5개 종류만 추출
전체 ePillID는 13,532장, 4,902종의 저샷 벤치마크다. 이번 실습은 시각적으로 구분 가능한 5종만 고른다. 각 종류는 소비자 사진 5장과 기준 사진 앞·뒤 2장으로 구성된다.

- train: 기준 사진 2장 + 소비자 사진 3장 = 종류당 5장
- val: 소비자 사진 1장 = 종류당 1장
- test: 소비자 사진 1장 = 종류당 1장

In [ ]:
DATA_URL = 'https://github.com/usuyama/ePillID-benchmark/releases/download/ePillID_data_v1.0/ePillID_data.zip'
ZIP_PATH = WORK_DIR / 'ePillID_data.zip'
SELECTED_PILL_TYPES = [
    '00003-4222-16_F215F93F',  # tan oblong tablet
    '00004-0800-85_491E24C1',  # white/yellow capsule
    '00006-0277-54_16160B30',  # orange round tablet
    '00023-9350-07_D4196A7B',  # orange/white capsule
    '00029-3159-13_E418F247',  # pink shield tablet
]
CLASS_MAP = {pill_id: 'pill_' + pill_id.split('_')[0].replace('-', '_') for pill_id in SELECTED_PILL_TYPES}

WORK_DIR.mkdir(parents=True, exist_ok=True)
if not ZIP_PATH.exists():
    print('Downloading ePillID release (about 160 MB)...')
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

manifest_rows = []
with zipfile.ZipFile(ZIP_PATH) as zf:
    with zf.open('ePillID_data/all_labels.csv') as f:
        labels = pd.read_csv(f)

    for pill_id in SELECTED_PILL_TYPES:
        class_name = CLASS_MAP[pill_id]
        rows = labels[labels['pilltype_id'] == pill_id].copy()
        ref_rows = rows[rows['is_ref'] == True].sort_values('images')
        consumer_rows = rows[rows['is_ref'] == False].sort_values('images')
        assert len(ref_rows) == 2 and len(consumer_rows) == 5, (pill_id, len(ref_rows), len(consumer_rows))

        split_rows = {
            'train': pd.concat([ref_rows, consumer_rows.iloc[:3]]),
            'val': consumer_rows.iloc[3:4],
            'test': consumer_rows.iloc[4:5],
        }

        for split, selected_rows in split_rows.items():
            target_dir = DATASET_DIR / split / class_name
            target_dir.mkdir(parents=True, exist_ok=True)
            for _, row in selected_rows.iterrows():
                member = f"ePillID_data/classification_data/{row['image_path']}"
                target = target_dir / Path(row['image_path']).name
                with zf.open(member) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
                manifest_rows.append({
                    'split': split, 'class_name': class_name, 'pilltype_id': pill_id,
                    'is_reference': bool(row['is_ref']), 'is_front': bool(row['is_front']),
                    'source_path': row['image_path'], 'local_path': str(target),
                })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUTPUT_DIR / 'classification_split_manifest.csv', index=False)
print(manifest.groupby(['split', 'class_name']).size().unstack(fill_value=0))

## 3. 데이터 원본 확인
각 행은 한 종류이며, 왼쪽은 학습에 들어간 기준 이미지, 오른쪽은 실제 촬영 조건에 가까운 test 소비자 이미지다.

In [ ]:
fig, axes = plt.subplots(len(SELECTED_PILL_TYPES), 2, figsize=(8, 15))
for i, pill_id in enumerate(SELECTED_PILL_TYPES):
    class_name = CLASS_MAP[pill_id]
    ref_path = Path(manifest[(manifest.class_name == class_name) & (manifest.is_reference == True)].iloc[0].local_path)
    test_path = Path(manifest[(manifest.class_name == class_name) & (manifest.split == 'test')].iloc[0].local_path)
    for j, (img_path, label) in enumerate([(ref_path, 'Reference / train'), (test_path, 'Consumer / test')]):
        image = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        axes[i, j].imshow(image)
        axes[i, j].set_title(f'{class_name}\n{label}', fontsize=10)
        axes[i, j].axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'classification_dataset_samples.png', dpi=180, bbox_inches='tight')
plt.show()

## 4. YOLOv8n-cls fine-tuning
ImageNet 사전학습 분류 모델의 가중치를 5종 알약 데이터에 맞게 갱신한다. 데이터가 매우 작기 때문에 30 epoch 결과도 과적합 가능성이 크며 test 5장 성능을 일반화 성능으로 보지 않는다.

In [ ]:
classifier = YOLO('yolov8n-cls.pt')
train_results = classifier.train(
    data=str(DATASET_DIR),
    epochs=30,
    imgsz=224,
    batch=16,
    device=DEVICE,
    project=str(WORK_DIR / 'runs'),
    name='epillid_5class_yolov8n_cls',
    exist_ok=True,
    seed=SEED,
    deterministic=True,
    dropout=0.2,
    patience=15,
    plots=True,
)

RUN_DIR = Path(train_results.save_dir)
BEST_CLS = RUN_DIR / 'weights' / 'best.pt'
print('best classifier:', BEST_CLS)

## 5. 별도 test 5장 평가
각 종류에서 학습과 validation에 쓰지 않은 소비자 이미지 1장씩만 평가한다. Top-1 accuracy와 macro F1, confusion matrix를 기록한다.

In [ ]:
best_classifier = YOLO(str(BEST_CLS))
test_rows = manifest[manifest.split == 'test'].sort_values('class_name').reset_index(drop=True)
y_true, y_pred, confidences = [], [], []

for _, row in test_rows.iterrows():
    result = best_classifier.predict(row.local_path, verbose=False)[0]
    pred_idx = int(result.probs.top1)
    pred_name = result.names[pred_idx]
    confidence = float(result.probs.top1conf.cpu())
    y_true.append(row.class_name)
    y_pred.append(pred_name)
    confidences.append(confidence)

class_names = sorted(test_rows.class_name.unique())
accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, labels=class_names, average='macro', zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=class_names)
report = classification_report(y_true, y_pred, labels=class_names, output_dict=True, zero_division=0)

metrics = {
    'top1_accuracy': round(float(accuracy), 6),
    'macro_f1': round(float(macro_f1), 6),
    'test_images': len(test_rows),
    'classes': len(class_names),
    'model': 'yolov8n-cls.pt',
    'epochs': 30,
    'image_size': 224,
    'seed': SEED,
    'ultralytics_version': ultralytics_version,
}
(OUTPUT_DIR / 'classification_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'classification_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
pd.DataFrame({'true': y_true, 'pred': y_pred, 'confidence': confidences, 'path': test_rows.local_path}).to_csv(OUTPUT_DIR / 'classification_predictions.csv', index=False)
print(json.dumps(metrics, indent=2))

In [ ]:
fig, axes = plt.subplots(1, len(test_rows), figsize=(18, 4))
for ax, (_, row), pred, conf in zip(axes, test_rows.iterrows(), y_pred, confidences):
    image = cv2.cvtColor(cv2.imread(row.local_path), cv2.COLOR_BGR2RGB)
    correct = pred == row.class_name
    ax.imshow(image)
    ax.set_title(f'True: {row.class_name}\nPred: {pred} ({conf:.2f})', color='green' if correct else 'red', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'classification_test_predictions.png', dpi=180, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Test confusion matrix | accuracy={accuracy:.1%}, macro F1={macro_f1:.3f}')
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'classification_confusion_matrix.png', dpi=180, bbox_inches='tight')
plt.show()

## 6. Detection → crop → Classification 연결 코드
아래 함수가 실제 2단계 구조다. detector가 bbox를 찾고, OpenCV로 잘라낸 각 crop을 classifier가 한 종류로 분류한다.

**중요:** 기존 Medical Pills detector는 노란 블리스터 알약 도메인이고, 이번 ePillID classifier는 개별 처방약 사진 도메인이다. 두 모델을 바로 연결하면 classifier가 항상 5종 중 하나를 강제로 출력하므로 실제 시스템으로 평가할 수 없다. 통합하려면 동일한 5종 알약이 여러 개 놓인 장면과 종류별 bbox 라벨이 필요하다.

In [ ]:
def detect_crop_classify(image_path, detector, classifier, detect_conf=0.25):
    image_bgr = cv2.imread(str(image_path))
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    detection = detector.predict(image_rgb, conf=detect_conf, verbose=False)[0]
    outputs = []

    for box, det_conf in zip(detection.boxes.xyxy.cpu().numpy(), detection.boxes.conf.cpu().numpy()):
        x1, y1, x2, y2 = box.astype(int)
        crop_rgb = image_rgb[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
        if crop_rgb.size == 0:
            continue
        cls_result = classifier.predict(crop_rgb, verbose=False)[0]
        cls_idx = int(cls_result.probs.top1)
        outputs.append({
            'bbox_xyxy': [int(x1), int(y1), int(x2), int(y2)],
            'detection_confidence': float(det_conf),
            'pill_type': cls_result.names[cls_idx],
            'classification_confidence': float(cls_result.probs.top1conf.cpu()),
        })
    return outputs

print('Pipeline function defined. Unified typed-bbox data is required before meaningful end-to-end evaluation.')

## 7. 결과 파일 정리
학습 곡선, 최적 가중치, test 예측, confusion matrix를 ZIP으로 묶는다.

In [ ]:
for source, target_name in [
    (BEST_CLS, 'best_classifier.pt'),
    (RUN_DIR / 'results.png', 'classification_training_curves.png'),
    (RUN_DIR / 'confusion_matrix.png', 'classification_val_confusion_matrix.png'),
]:
    if source.exists():
        shutil.copy2(source, OUTPUT_DIR / target_name)

archive_base = WORK_DIR / 'week3_classification_artifacts'
shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_DIR)

print('outputs:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f'- {path.name} ({path.stat().st_size / 1024:.1f} KB)')
print('archive:', archive_base.with_suffix('.zip'))

try:
    from google.colab import files
    files.download(str(archive_base.with_suffix('.zip')))
except ImportError:
    pass